# Clean the CHSP by LGA table from Excel

This notebook builds a simple data pipeline for the uploaded Excel file and only reads the required worksheet.

## Cleaning rules
- read only the `Table 1.1 (LGA)` worksheet
- detect the real header row from the Excel sheet
- keep only the 3 required columns
- drop completely empty rows
- trim and normalize whitespace in text fields
- convert blank strings to missing values
- rename the columns for psql / analysis
- remove commas from totals and convert them to integers
- remove repeated header rows if they appear inside the data
- remove exact duplicate rows

At the end, the final cleaned dataframe is available as `clean_df`.


In [32]:
from pathlib import Path
import pandas as pd

INPUT_XLSX = Path("input/gen_data_aged_care_demand.xlsx")
SHEET_NAME = "Table 1.1 (LGA)"

INPUT_XLSX, SHEET_NAME

(PosixPath('input/gen_data_aged_care_demand.xlsx'), 'Table 1.1 (LGA)')

In [33]:
raw_df = pd.read_excel(
    INPUT_XLSX,
    sheet_name=SHEET_NAME,
    header=None
)

raw_df.head(10)

,0,1,2
0,Table 1.1: Number of people using home support...,NaN,NaN
1,Return to contents,NaN,NaN
2,LGA (2017) code,LGA (2017) name,Total
3,10050,Albury (C),2186
4,10130,Armidale Regional (A),1273
5,10250,Ballina (A),2653
6,10300,Balranald (A),160
7,10470,Bathurst Regional (A),1524
8,10500,Bayside (A),4186
9,10550,Bega Valley (A),1611


In [34]:
def clean_aged_care_demand(input_path: Path, sheet_name: str = "Table 1.1 (LGA)"):
    # Read only the required worksheet without assuming the header row position.
    df = pd.read_excel(input_path, sheet_name=sheet_name, header=None)

    # Find the real header row by matching the first column name.
    header_mask = (
        df.iloc[:, 0].astype("string").str.contains(r"^LGA \(2017\) code$", na=False)
    )

    if not header_mask.any():
        raise ValueError("Could not find the header row for the LGA table.")

    header_row_idx = header_mask[header_mask].index[0]

    # Rebuild the dataframe using the detected header row.
    df = df.iloc[header_row_idx + 1:].copy()
    df.columns = df.iloc[0:0].reindex(columns=df.columns).columns  # keep column count stable
    df.columns = pd.Index(pd.read_excel(input_path, sheet_name=sheet_name, header=None).iloc[header_row_idx].tolist())

    # Keep only the required columns from the worksheet.
    keep_columns = ["LGA (2017) code", "LGA (2017) name", "Total"]
    df = df[keep_columns].copy()

    # Drop completely empty rows.
    df = df.dropna(axis=0, how="all")

    # Rename columns.
    df = df.rename(columns={
        "LGA (2017) code": "lga_code_2017",
        "LGA (2017) name": "lga_name_2017",
        "Total": "total_people_using_home_support",
    })

    # Clean text columns: normalize whitespace and convert blanks to missing values.
    text_cols = ["lga_code_2017", "lga_name_2017", "total_people_using_home_support"]
    for col in text_cols:
        s = df[col].astype("string")
        s = s.str.replace(r"\s+", " ", regex=True).str.strip()
        s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        df[col] = s

    # Remove repeated header rows if they appear in the body.
    df = df[df["lga_code_2017"] != "LGA (2017) code"].copy()

    #  Treat the code as strings.
    df["lga_code_2017"] = (
        pd.to_numeric(df["lga_code_2017"], errors="coerce")
        .astype("Int64").astype("string")
    )

    df["total_people_using_home_support"] = (
        df["total_people_using_home_support"]
        .str.replace(",", "", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
        .astype("Int64")
    )

    # Final row filtering: keep rows with a valid LGA code and name.
    df = df[df["lga_code_2017"].notna()].copy()
    df = df[df["lga_name_2017"].notna()].copy()
    
    df["lga_type_2017"] = df["lga_name_2017"].str.extract(r"\((.*?)\)")
    df["lga_name_2017"] = df["lga_name_2017"].str.replace(r"\s*\(.*?\)", "", regex=True)

    # Remove exact duplicates.
    rows_before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    duplicates_removed = rows_before - len(df)

    # Return a summary of the cleaning process.
    summary = {
        "sheet_name": sheet_name,
        "output_rows": len(df),
        "duplicates_removed": duplicates_removed,
        "column_count": df.shape[1],
        "null_lga_code_rows": int(df["lga_code_2017"].isna().sum()),
        "null_lga_name_rows": int(df["lga_name_2017"].isna().sum()),
        "null_total_rows": int(df["total_people_using_home_support"].isna().sum()),
    }

    return df, summary

In [35]:
clean_df, summary = clean_aged_care_demand(INPUT_XLSX, SHEET_NAME)
summary

{'sheet_name': 'Table 1.1 (LGA)',
 'output_rows': 536,
 'duplicates_removed': 0,
 'column_count': 4,
 'null_lga_code_rows': 0,
 'null_lga_name_rows': 0,
 'null_total_rows': 0}

In [36]:
clean_df.head()

,lga_code_2017,lga_name_2017,total_people_using_home_support,lga_type_2017
0,10050,Albury,2186,C
1,10130,Armidale Regional,1273,A
2,10250,Ballina,2653,A
3,10300,Balranald,160,A
4,10470,Bathurst Regional,1524,A


In [37]:
print(f"Blank rows remaining: {int(clean_df.isna().all(axis=1).sum())}")
print(f"Exact duplicates remaining: {int(clean_df.duplicated().sum())}")
print(f"Null lga_code_2017 rows: {int(clean_df['lga_code_2017'].isna().sum())}")
print(f"Null lga_name_2017 rows: {int(clean_df['lga_name_2017'].isna().sum())}")
print(f"Data types:\n{clean_df.dtypes}")

Blank rows remaining: 0
Exact duplicates remaining: 0
Null lga_code_2017 rows: 0
Null lga_name_2017 rows: 0
Data types:
lga_code_2017                      string
lga_name_2017                      string
total_people_using_home_support     Int64
lga_type_2017                      string
dtype: object


# Insert Clean Data into DB

In [38]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

# -----------------------------
# 1. DATABASE CONNECTION
# -----------------------------
load_dotenv('.env.db', override=True)

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
)

engine = create_engine(DATABASE_URL)

# -----------------------------
# 2. CREATE TABLE FROM SCHEMA
# -----------------------------
schema_path = Path("schemas/aged_care_demand.sql")

with open(schema_path, "r", encoding="utf-8") as f:
    schema_sql = f.read()

with engine.begin() as conn:
    conn.execute(text(schema_sql))

print("Schema created or already exists.")
print(DB_NAME)

Schema created or already exists.
carelink


In [39]:
# -----------------------------
# 3. PREPARE DATAFRAME
# -----------------------------
df_insert = clean_df.copy()

target_columns = [
    "lga_code_2017",
    "lga_name_2017",
    "lga_type_2017",
    "total_people_using_home_support",
]

df_insert = df_insert[target_columns].copy()

# text columns (UPDATED: include lga_code_2017)
text_cols = [
    "lga_code_2017",
    "lga_name_2017",
    "lga_type_2017",
]

for col in text_cols:
    df_insert[col] = df_insert[col].astype("string").str.strip()

# integer columns (REMOVED lga_code_2017)
int_cols = [
    "total_people_using_home_support",
]

for col in int_cols:
    df_insert[col] = pd.to_numeric(df_insert[col], errors="coerce").astype("Int64")

In [40]:
# -----------------------------
# 4. INSERT INTO DATABASE
# -----------------------------
df_insert.to_sql(
    "aged_care_demand",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000,
)

print(f"Inserted {len(df_insert)} rows into aged_care_demand")

# -----------------------------
# 5. VALIDATE
# -----------------------------
with engine.begin() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM aged_care_demand"))
    row_count = result.scalar()

print("Rows in table:", row_count)

Inserted 536 rows into aged_care_demand
Rows in table: 536
